In [1]:
from pathlib import Path
import duckdb
import pandas as pd
import os
import pyarrow.parquet as pq
import urllib.request

In [2]:
data_dir = Path("../data/raw")
files = sorted(data_dir.glob("yellow_tripdata_2025-*.parquet"))

print("Số file:", len(files))

Số file: 12


In [3]:
df = pd.read_parquet(files[0])

display(df.head())

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0


In [4]:
total_size_mb = sum(os.path.getsize(f) for f in files) / (1024 * 1024)

con = duckdb.connect()
stats_5v = con.execute("""
    SELECT 
        COUNT(*) AS tong_so_dong,
        ROUND(COUNT(*) / 365.0, 0) AS trung_binh_moi_ngay,
        ROUND(COUNT(*) / (365.0 * 24 * 60), 0) AS trung_binh_moi_phut,
        SUM(CASE WHEN passenger_count IS NULL THEN 1 ELSE 0 END) AS so_dong_null_khach,
        SUM(CASE WHEN trip_distance <= 0 OR fare_amount <= 0 OR tpep_dropoff_datetime <= tpep_pickup_datetime THEN 1 ELSE 0 END) AS so_dong_loi_logic
    FROM read_parquet('../data/raw/yellow_tripdata_2025-*.parquet')
""").df()

print(f"Tổng dung lượng 12 file Parquet: {total_size_mb:.2f} MB")
display(stats_5v)

Tổng dung lượng 12 file Parquet: 791.52 MB


,tong_so_dong,trung_binh_moi_ngay,trung_binh_moi_phut,so_dong_null_khach,so_dong_loi_logic
0,48722602,133487.0,93.0,11611894.0,4550667.0


In [5]:
schema_list = []
volume_list = []

for f in files:
    # Đọc metadata và schema trực tiếp từ Parquet (cực nhanh)
    parquet_file = pq.ParquetFile(f)
    schema = parquet_file.schema_arrow
    
    schema_list.append({col.name: str(col.type) for col in schema})
    volume_list.append({
        'Tháng': f.stem.replace('yellow_tripdata_', ''),
        'Số bản ghi': parquet_file.metadata.num_rows,
        'Số cột': parquet_file.metadata.num_columns
    })

df_volume = pd.DataFrame(volume_list)
df_schemas = pd.DataFrame(schema_list, index=[f.stem[-7:] for f in files])

print("1. BẢNG THỐNG KÊ QUY MÔ 12 THÁNG:")
display(df_volume)

print("\n2. KIỂM TRA SCHEMA GIỮA 12 THÁNG:")
display(df_schemas.T)

1. BẢNG THỐNG KÊ QUY MÔ 12 THÁNG:


,Tháng,Số bản ghi,Số cột
0,2025-01,3475226,20
1,2025-02,3577543,20
2,2025-03,4145257,20
3,2025-04,3970553,20
4,2025-05,4591845,20
5,2025-06,4322960,20
6,2025-07,3898963,20
7,2025-08,3574091,20
8,2025-09,4251015,20
9,2025-10,4428699,20



2. KIỂM TRA SCHEMA GIỮA 12 THÁNG:


,2025-01,2025-02,2025-03,2025-04,2025-05,2025-06,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12
VendorID,int32,int32,int32,int32,int32,int32,int32,int32,int32,int32,int32,int32
tpep_pickup_datetime,timestamp[us],timestamp[us],timestamp[us],timestamp[us],timestamp[us],timestamp[us],timestamp[us],timestamp[us],timestamp[us],timestamp[us],timestamp[us],timestamp[us]
tpep_dropoff_datetime,timestamp[us],timestamp[us],timestamp[us],timestamp[us],timestamp[us],timestamp[us],timestamp[us],timestamp[us],timestamp[us],timestamp[us],timestamp[us],timestamp[us]
passenger_count,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64
trip_distance,double,double,double,double,double,double,double,double,double,double,double,double
RatecodeID,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64
store_and_fwd_flag,large_string,large_string,large_string,large_string,large_string,large_string,large_string,large_string,large_string,large_string,large_string,large_string
PULocationID,int32,int32,int32,int32,int32,int32,int32,int32,int32,int32,int32,int32
DOLocationID,int32,int32,int32,int32,int32,int32,int32,int32,int32,int32,int32,int32
payment_type,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64,int64


In [6]:
con = duckdb.connect()

# Đếm tỷ lệ % NULL của các cột có chứa NULL theo từng tháng
df_missing = con.execute(r"""
    SELECT 
        regexp_extract(filename, '2025-\d{2}', 0) AS thang,
        COUNT(*) AS tong_dong,
        COUNT(*) - COUNT(passenger_count) AS null_passenger_count,
        COUNT(*) - COUNT(RatecodeID) AS null_ratecode,
        COUNT(*) - COUNT(store_and_fwd_flag) AS null_store_flag,
        COUNT(*) - COUNT(congestion_surcharge) AS null_congestion,
        COUNT(*) - COUNT(Airport_fee) AS null_airport_fee,
        ROUND((COUNT(*) - COUNT(passenger_count)) * 100.0 / COUNT(*), 2) AS pct_null
    FROM read_parquet('../data/raw/yellow_tripdata_2025-*.parquet', filename=true)
    GROUP BY thang
    ORDER BY thang
""").df()

display(df_missing)

,thang,tong_dong,null_passenger_count,null_ratecode,null_store_flag,null_congestion,null_airport_fee,pct_null
0,2025-01,3475226,540149,540149,540149,540149,540149,15.54
1,2025-02,3577543,806937,806937,806937,806937,806937,22.56
2,2025-03,4145257,916663,916663,916663,916663,916663,22.11
3,2025-04,3970553,745730,745730,745730,745730,745730,18.78
4,2025-05,4591845,1196176,1196176,1196176,1196176,1196176,26.05
5,2025-06,4322960,1212946,1212946,1212946,1212946,1212946,28.06
6,2025-07,3898963,1038755,1038755,1038755,1038755,1038755,26.64
7,2025-08,3574091,886234,886234,886234,886234,886234,24.80
8,2025-09,4251015,1067195,1067195,1067195,1067195,1067195,25.10
9,2025-10,4428699,990887,990887,990887,990887,990887,22.37


In [7]:
df_quality_checks = con.execute("""
    SELECT 
        regexp_extract(filename, '2025-\d{2}', 0) AS thang,
        -- Mục 3: Kiểm tra thời gian
        SUM(CASE WHEN tpep_dropoff_datetime < tpep_pickup_datetime THEN 1 ELSE 0 END) AS tg_ket_thuc_truoc_bat_dau,
        SUM(CASE WHEN tpep_dropoff_datetime = tpep_pickup_datetime THEN 1 ELSE 0 END) AS thoi_luong_bang_0,
        SUM(CASE WHEN strftime(tpep_pickup_datetime, '%Y-%m') != regexp_extract(filename, '2025-\d{2}', 0) THEN 1 ELSE 0 END) AS sai_thang_nam,
        
        -- Mục 4: Kiểm tra quãng đường
        SUM(CASE WHEN trip_distance < 0 THEN 1 ELSE 0 END) AS quang_duong_am,
        SUM(CASE WHEN trip_distance = 0 THEN 1 ELSE 0 END) AS quang_duong_bang_0,
        SUM(CASE WHEN trip_distance > 500 THEN 1 ELSE 0 END) AS quang_duong_bat_thuong_lon,
        
        -- Mục 5: Kiểm tra tiền cước
        SUM(CASE WHEN fare_amount < 0 THEN 1 ELSE 0 END) AS tien_cuoc_am,
        SUM(CASE WHEN fare_amount = 0 THEN 1 ELSE 0 END) AS tien_cuoc_bang_0,
        SUM(CASE WHEN total_amount <= 0 THEN 1 ELSE 0 END) AS tong_tien_am_hoac_0
    FROM read_parquet('../data/raw/yellow_tripdata_2025-*.parquet', filename=true)
    GROUP BY thang
    ORDER BY thang
""").df()

display(df_quality_checks)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,thang,tg_ket_thuc_truoc_bat_dau,thoi_luong_bang_0,sai_thang_nam,quang_duong_am,quang_duong_bang_0,quang_duong_bat_thuong_lon,tien_cuoc_am,tien_cuoc_bang_0,tong_tien_am_hoac_0
0,2025-01,124.0,1927.0,22.0,0.0,90893.0,118.0,144118.0,1398.0,63596.0
1,2025-02,93.0,5071.0,31.0,0.0,99771.0,141.0,182655.0,1441.0,55621.0
2,2025-03,81.0,22199.0,33.0,0.0,103722.0,168.0,208724.0,1654.0,69287.0
3,2025-04,163.0,34606.0,7.0,0.0,91439.0,259.0,186720.0,1451.0,73222.0
4,2025-05,101.0,64171.0,24.0,0.0,141121.0,298.0,325266.0,2109.0,113064.0
5,2025-06,231.0,68352.0,20.0,0.0,134896.0,200.0,275565.0,1962.0,74888.0
6,2025-07,1.0,56063.0,7.0,0.0,123774.0,171.0,247088.0,1370.0,77038.0
7,2025-08,2.0,47831.0,17.0,0.0,105184.0,134.0,261191.0,1444.0,88458.0
8,2025-09,0.0,57179.0,7.0,0.0,124483.0,149.0,252718.0,1706.0,79109.0
9,2025-10,2.0,67966.0,13.0,0.0,125599.0,150.0,321307.0,2076.0,109312.0


In [ ]:
zone_file = Path("../data/raw/taxi_zone_lookup.csv")
df_zones = pd.read_csv(zone_file)
display(df_zones.head())

df_zone_check = con.execute(r"""
    SELECT 
        COUNT(*) AS tong_so_chuyen,
        SUM(CASE WHEN pu_zone.LocationID IS NULL THEN 1 ELSE 0 END) AS pu_khong_ton_tai_trong_lookup,
        SUM(CASE WHEN do_zone.LocationID IS NULL THEN 1 ELSE 0 END) AS do_khong_ton_tai_trong_lookup,
        SUM(CASE WHEN t.PULocationID IN (264, 265) OR t.DOLocationID IN (264, 265) THEN 1 ELSE 0 END) AS ma_khu_vuc_unknown_264_265
    FROM read_parquet('../data/raw/yellow_tripdata_2025-*.parquet') t
    LEFT JOIN read_csv_auto('../data/raw/taxi_zone_lookup.csv') pu_zone ON t.PULocationID = pu_zone.LocationID
    LEFT JOIN read_csv_auto('../data/raw/taxi_zone_lookup.csv') do_zone ON t.DOLocationID = do_zone.LocationID
""").df()

display(df_zone_check)

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


,tong_so_chuyen,pu_khong_ton_tai_trong_lookup,do_khong_ton_tai_trong_lookup,ma_khu_vuc_unknown_264_265
0,48722602,0.0,0.0,371657.0


In [9]:
df_vendor = con.execute("""
    SELECT
        VendorID,
        COUNT(*) AS tong_chuyen,
        COUNT(*) FILTER (
            WHERE passenger_count IS NULL
        ) AS thieu_hanh_khach,
        ROUND(
            COUNT(*) FILTER (
                WHERE passenger_count IS NULL
            ) * 100.0 / COUNT(*), 2
        ) AS ty_le_thieu
    FROM read_parquet('../data/raw/yellow_tripdata_2025-*.parquet')
    GROUP BY VendorID
    ORDER BY VendorID
""").df()

display(df_vendor)

,VendorID,tong_chuyen,thieu_hanh_khach,ty_le_thieu
0,1,9586873,1579612,16.48
1,2,38575846,10008300,25.94
2,6,23982,23982,100.00
3,7,535901,0,0.00


In [10]:
df_fare = con.execute("""
    SELECT
        payment_type,
        COUNT(*) AS tong_chuyen,
        COUNT(*) FILTER (
            WHERE fare_amount < 0
        ) AS tien_cuoc_am
    FROM read_parquet('../data/raw/yellow_tripdata_2025-*.parquet')
    GROUP BY payment_type
    ORDER BY payment_type
""").df()

display(df_fare)

,payment_type,tong_chuyen,tien_cuoc_am
0,0,11611894,2047089
1,1,31054000,1130
2,2,4654345,186208
3,3,308147,100166
4,4,1094213,514027
5,5,3,0


In [11]:
df_null_check = con.execute("""
    SELECT
        payment_type,
        COUNT(*) AS tong_chuyen,
        COUNT(*) FILTER (
            WHERE passenger_count IS NULL
        ) AS null_passenger,
        COUNT(*) FILTER (
            WHERE passenger_count IS NULL
              AND RatecodeID IS NULL
              AND store_and_fwd_flag IS NULL
              AND congestion_surcharge IS NULL
              AND Airport_fee IS NULL
        ) AS null_ca_5_cot
    FROM read_parquet('../data/raw/yellow_tripdata_2025-*.parquet')
    GROUP BY payment_type
    ORDER BY payment_type
""").df()

display(df_null_check)

,payment_type,tong_chuyen,null_passenger,null_ca_5_cot
0,0,11611894,11611894,11611894
1,1,31054000,0,0
2,2,4654345,0,0
3,3,308147,0,0
4,4,1094213,0,0
5,5,3,0,0


In [12]:
columns = pq.read_schema(files[0]).names

query = ",\n".join(
    f'COUNT(*) FILTER (WHERE "{col}" IS NULL) AS "{col}"'
    for col in columns
)

df_all_null = con.execute(f"""
    SELECT {query}
    FROM read_parquet('../data/raw/yellow_tripdata_2025-*.parquet')
""").df()

display(df_all_null.T)

,0
VendorID,0
tpep_pickup_datetime,0
tpep_dropoff_datetime,0
passenger_count,11611894
trip_distance,0
RatecodeID,11611894
store_and_fwd_flag,11611894
PULocationID,0
DOLocationID,0
payment_type,0


In [ ]:
# Kiểm tra tiền cước âm có phải tổng tiền âm không
fare_check = con.execute("""
    SELECT
        payment_type,
        COUNT(*) AS negative_fare,
        COUNT(*) FILTER (
            WHERE total_amount < 0
        ) AS negative_total,
        COUNT(*) FILTER (
            WHERE total_amount >= 0
        ) AS non_negative_total
    FROM read_parquet('../data/raw/yellow_tripdata_2025-*.parquet')
    WHERE fare_amount < 0
    GROUP BY payment_type
    ORDER BY payment_type
""").df()

display(fare_check)

,payment_type,negative_fare,negative_total,non_negative_total
0,0,2047089,167217,1879872
1,1,1130,1130,0
2,2,186208,186208,0
3,3,100166,100166,0
4,4,514027,514026,1


In [15]:
# Kiểm tra thời gian âm có tập trung vào ngày đổi giờ không
time_check = con.execute("""
    SELECT
        COUNT(*) AS total_invalid_time,

        COUNT(*) FILTER (
            WHERE CAST(tpep_pickup_datetime AS DATE)
                  = DATE '2025-11-02'
              AND EXTRACT(HOUR FROM tpep_pickup_datetime) = 1
              AND EXTRACT(HOUR FROM tpep_dropoff_datetime) = 1
        ) AS possible_dst

    FROM read_parquet('../data/raw/yellow_tripdata_2025-*.parquet')

    WHERE tpep_dropoff_datetime < tpep_pickup_datetime
""").df()

display(time_check)

,total_invalid_time,possible_dst
0,2235,1431


In [16]:
# Kiểm tra quãng đường bằng 0 thuộc nhóm chuyến nào
distance_check = con.execute("""
    SELECT
        payment_type,
        COUNT(*) AS zero_distance,

        COUNT(*) FILTER (
            WHERE total_amount > 0
        ) AS positive_total,

        COUNT(*) FILTER (
            WHERE tpep_dropoff_datetime > tpep_pickup_datetime
        ) AS positive_duration

    FROM read_parquet('../data/raw/yellow_tripdata_2025-*.parquet')

    WHERE trip_distance = 0
    GROUP BY payment_type
    ORDER BY payment_type
""").df()

display(distance_check)

,payment_type,zero_distance,positive_total,positive_duration
0,0,904912,888393,904161
1,1,259148,258350,251289
2,2,108477,96511,103001
3,3,58142,41054,56970
4,4,72277,42875,71985
5,5,2,0,1


In [17]:
zero_check = con.execute("""
    SELECT
        payment_type,
        COUNT(*) AS zero_distance,

        COUNT(*) FILTER (
            WHERE total_amount > 0
              AND tpep_dropoff_datetime > tpep_pickup_datetime
        ) AS positive_time_and_amount

    FROM read_parquet(
        '../data/raw/yellow_tripdata_2025-*.parquet'
    )
    WHERE trip_distance = 0
    GROUP BY payment_type
    ORDER BY payment_type
""").df()

display(zero_check)

,payment_type,zero_distance,positive_time_and_amount
0,0,904912,887647
1,1,259148,250500
2,2,108477,91290
3,3,58142,39948
4,4,72277,42646
5,5,2,0
